# 🔢 MNIST — Fully Connected Network vs CNN
## Comparaison de deux architectures pour la classification de chiffres manuscrits

| | FCN | CNN |
|---|---|---|
| Entrée | 784 pixels aplatis | 28×28×1 image 2D |
| Couches | Dense uniquement | Conv2D + MaxPool + Dense |
| Exploite la spatialité | ❌ | ✅ |
| Accuracy attendue | ~97-98% | ~99%+ |

---

## 📦 Section 1 — Setup & Chargement MNIST

In [ ]:
!pip install -q tensorflow matplotlib seaborn scikit-learn numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
tf.random.set_seed(42)

print(f'✅ TensorFlow : {tf.__version__}')
print(f'✅ GPU dispo  : {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# === ÉTAPE 1 : Charger le dataset MNIST ===
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

# Afficher les shapes
print('=== SHAPES DU DATASET MNIST ===')
print(f'  X_train : {X_train_raw.shape}  → {X_train_raw.shape[0]:,} images de {X_train_raw.shape[1]}×{X_train_raw.shape[2]} pixels')
print(f'  y_train : {y_train_raw.shape}')
print(f'  X_test  : {X_test_raw.shape}   → {X_test_raw.shape[0]:,} images de {X_test_raw.shape[1]}×{X_test_raw.shape[2]} pixels')
print(f'  y_test  : {y_test_raw.shape}')
print(f'\n  Valeurs pixel  : min={X_train_raw.min()}, max={X_train_raw.max()}')
print(f'  Classes        : {np.unique(y_train_raw)}  (chiffres 0 à 9)')
print(f'  Dtype images   : {X_train_raw.dtype}')
print(f'  Dtype labels   : {y_train_raw.dtype}')

In [ ]:
# Visualisation de quelques exemples
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
for digit in range(10):
    idx = np.where(y_train_raw == digit)[0][0]
    axes[0][digit].imshow(X_train_raw[idx], cmap='gray')
    axes[0][digit].set_title(str(digit), fontsize=13, fontweight='bold', color='#185FA5')
    axes[0][digit].axis('off')
    idx2 = np.where(y_train_raw == digit)[0][1]
    axes[1][digit].imshow(X_train_raw[idx2], cmap='gray')
    axes[1][digit].axis('off')
plt.suptitle('Exemples MNIST — 2 images par chiffre (0–9)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Distribution des classes
fig, ax = plt.subplots(figsize=(10, 3))
unique, counts = np.unique(y_train_raw, return_counts=True)
ax.bar(unique, counts, color='#185FA5', edgecolor='white')
ax.set_xticks(range(10))
ax.set_title('Distribution des classes dans le train set', fontweight='bold')
ax.set_xlabel('Chiffre'); ax.set_ylabel('Nombre d\'images')
for i, (d, c) in enumerate(zip(unique, counts)):
    ax.text(d, c + 50, f'{c:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 🧠 Section 2 — Fully Connected Neural Network (FCN)
### Étape 2 : Preprocessing pour le FCN

In [ ]:
# === ÉTAPE 2 : Preprocessing FCN ===

# a) Aplatir les images : 28×28 → 784
X_train_flat = X_train_raw.reshape(X_train_raw.shape[0], -1)  # (60000, 784)
X_test_flat  = X_test_raw.reshape(X_test_raw.shape[0], -1)    # (10000, 784)
print(f'✅ Aplatissement : {X_train_raw.shape} → {X_train_flat.shape}')

# b) Normalisation : pixels 0-255 → 0.0-1.0
X_train_flat = X_train_flat.astype('float32') / 255.0
X_test_flat  = X_test_flat.astype('float32')  / 255.0
print(f'✅ Normalisation : pixels dans [{X_train_flat.min():.1f}, {X_train_flat.max():.1f}]')

# c) One-hot encoding des labels
y_train_ohe = to_categorical(y_train_raw, num_classes=10)
y_test_ohe  = to_categorical(y_test_raw,  num_classes=10)
print(f'✅ One-hot encoding : {y_train_raw.shape} → {y_train_ohe.shape}')
print(f'   Exemple : chiffre {y_train_raw[0]} → {y_train_ohe[0]}')

print(f'\n=== SHAPES FINALES (FCN) ===')
print(f'  X_train_flat : {X_train_flat.shape}')
print(f'  X_test_flat  : {X_test_flat.shape}')
print(f'  y_train_ohe  : {y_train_ohe.shape}')
print(f'  y_test_ohe   : {y_test_ohe.shape}')

In [ ]:
# === ÉTAPE 3 : Construire le FCN ===
fcn_model = keras.Sequential([
    # Couche d'entrée
    layers.Input(shape=(784,), name='input'),

    # Couche cachée 1 — 512 neurones, activation ReLU
    layers.Dense(512, activation='relu', name='dense_1'),
    layers.Dropout(0.2, name='dropout_1'),

    # Couche cachée 2 — 256 neurones, activation ReLU
    layers.Dense(256, activation='relu', name='dense_2'),
    layers.Dropout(0.2, name='dropout_2'),

    # Couche de sortie — 10 neurones (un par chiffre), Softmax
    layers.Dense(10, activation='softmax', name='output')
], name='FCN_Model')

# Compiler le modèle
fcn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fcn_model.summary()

In [ ]:
# === ÉTAPE 3 (suite) : Entraîner le FCN ===
print('🏋️  Entraînement du FCN...')

fcn_history = fcn_model.fit(
    X_train_flat, y_train_ohe,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# Évaluation
fcn_loss, fcn_acc = fcn_model.evaluate(X_test_flat, y_test_ohe, verbose=0)
print(f'\n✅ FCN — Test Loss     : {fcn_loss:.4f}')
print(f'✅ FCN — Test Accuracy : {fcn_acc:.4f} ({fcn_acc*100:.2f}%)')

---
## 🏗️ Section 3 — Convolutional Neural Network (CNN)
### Étape 4 : Preprocessing pour le CNN

In [ ]:
# === ÉTAPE 4 : Preprocessing CNN ===

# a) Reshape : (60000, 28, 28) → (60000, 28, 28, 1)  [ajout du canal couleur]
X_train_cnn = X_train_raw.reshape(X_train_raw.shape[0], 28, 28, 1)
X_test_cnn  = X_test_raw.reshape(X_test_raw.shape[0], 28, 28, 1)
print(f'✅ Reshape : {X_train_raw.shape} → {X_train_cnn.shape}')

# b) Normalisation
X_train_cnn = X_train_cnn.astype('float32') / 255.0
X_test_cnn  = X_test_cnn.astype('float32')  / 255.0
print(f'✅ Normalisation : pixels dans [{X_train_cnn.min():.1f}, {X_train_cnn.max():.1f}]')

# c) One-hot encoding (réutilisé depuis le preprocessing FCN)
print(f'✅ One-hot encoding : déjà fait → {y_train_ohe.shape}')

print(f'\n=== SHAPES FINALES (CNN) ===')
print(f'  X_train_cnn  : {X_train_cnn.shape}  ← format (N, hauteur, largeur, canaux)')
print(f'  X_test_cnn   : {X_test_cnn.shape}')
print(f'\n  Pourquoi (28, 28, 1) ?')
print(f'  → Conv2D attend un tenseur 4D : (batch, height, width, channels)')
print(f'  → 1 canal car images en niveaux de gris (RGB aurait 3 canaux)')

In [ ]:
# === ÉTAPE 5 : Construire le CNN ===
cnn_model = keras.Sequential([
    # Bloc convolutif 1
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu',
                  input_shape=(28, 28, 1), name='conv_1'),
    layers.MaxPool2D(pool_size=(2, 2), name='pool_1'),

    # Bloc convolutif 2
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu', name='conv_2'),
    layers.MaxPool2D(pool_size=(2, 2), name='pool_2'),

    # Aplatissement pour passer aux couches Dense
    layers.Flatten(name='flatten'),

    # Couches fully connected
    layers.Dense(128, activation='relu', name='dense_1'),
    layers.Dropout(0.3, name='dropout'),

    # Couche de sortie
    layers.Dense(10, activation='softmax', name='output')
], name='CNN_Model')

# Compiler
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# Visualisation de ce que "voient" les filtres Conv2D
fig, axes = plt.subplots(2, 5, figsize=(14, 5))
sample_img = X_train_cnn[0:1]  # 1 image

# Après Conv1 (avant pooling)
feature_model = keras.Model(inputs=cnn_model.input,
                            outputs=cnn_model.get_layer('conv_1').output)
features = feature_model.predict(sample_img, verbose=0)  # (1, 26, 26, 32)

axes[0][0].imshow(X_train_raw[0], cmap='gray')
axes[0][0].set_title('Image originale\n(28×28)', fontweight='bold')
axes[0][0].axis('off')

for i in range(1, 5):
    axes[0][i].imshow(features[0, :, :, i*5], cmap='viridis')
    axes[0][i].set_title(f'Filtre Conv1 #{i*5}\n(26×26)', fontsize=9)
    axes[0][i].axis('off')

# Après Conv2
feature_model2 = keras.Model(inputs=cnn_model.input,
                             outputs=cnn_model.get_layer('conv_2').output)
features2 = feature_model2.predict(sample_img, verbose=0)  # (1, 5, 5, 64)

axes[1][0].imshow(X_train_raw[0], cmap='gray')
axes[1][0].set_title('Image originale', fontweight='bold')
axes[1][0].axis('off')

for i in range(1, 5):
    axes[1][i].imshow(features2[0, :, :, i*10], cmap='plasma')
    axes[1][i].set_title(f'Filtre Conv2 #{i*10}\n(5×5)', fontsize=9)
    axes[1][i].axis('off')

plt.suptitle('Feature Maps — Ce que le CNN "voit" à chaque couche',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === ÉTAPE 5 (suite) : Entraîner le CNN ===
print('🏋️  Entraînement du CNN...')

cnn_history = cnn_model.fit(
    X_train_cnn, y_train_ohe,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# Évaluation
cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test_ohe, verbose=0)
print(f'\n✅ CNN — Test Loss     : {cnn_loss:.4f}')
print(f'✅ CNN — Test Accuracy : {cnn_acc:.4f} ({cnn_acc*100:.2f}%)')

---
## ⚖️ Section 4 — Comparaison FCN vs CNN
### Étape 6 : Analyser les performances

In [ ]:
# === ÉTAPE 6 : Comparaison des performances ===

print('=' * 60)
print('          COMPARAISON FCN vs CNN — MNIST')
print('=' * 60)
print(f'  {"Métrique":<25} {"FCN":>10} {"CNN":>10}')
print('-' * 60)
print(f'  {"Test Accuracy":<25} {fcn_acc*100:>9.2f}% {cnn_acc*100:>9.2f}%')
print(f'  {"Test Loss":<25} {fcn_loss:>10.4f} {cnn_loss:>10.4f}')
print(f'  {"Erreurs / 10 000":<25} {int((1-fcn_acc)*10000):>10} {int((1-cnn_acc)*10000):>10}')
print(f'  {"Total Paramètres":<25} {fcn_model.count_params():>10,} {cnn_model.count_params():>10,}')
print(f'  {"Architecture":<25} {"Dense only":>10} {"Conv+Dense":>10}')
print(f'  {"Exploite spatialité":<25} {"Non":>10} {"Oui":>10}')
print('=' * 60)
print(f'\n  Amélioration CNN vs FCN : +{(cnn_acc - fcn_acc)*100:.2f}% accuracy')
print(f'  Réduction des erreurs   : {int((1-fcn_acc)*10000) - int((1-cnn_acc)*10000)} images supplémentaires correctes')

In [ ]:
# === COURBES D'APPRENTISSAGE CÔTE À CÔTE ===
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

epochs = range(1, 11)

# --- FCN ---
# Accuracy FCN
axes[0][0].plot(epochs, fcn_history.history['accuracy'],
                color='#185FA5', lw=2.5, marker='o', ms=6, label='Train')
axes[0][0].plot(epochs, fcn_history.history['val_accuracy'],
                color='#185FA5', lw=2.5, marker='s', ms=6,
                linestyle='--', alpha=0.7, label='Validation')
axes[0][0].set_title('FCN — Accuracy', fontweight='bold', fontsize=12)
axes[0][0].set_xlabel('Epoch'); axes[0][0].set_ylabel('Accuracy')
axes[0][0].legend()
axes[0][0].set_ylim(0.9, 1.01)
axes[0][0].axhline(fcn_acc, color='gray', lw=1, linestyle=':',
                    label=f'Test={fcn_acc:.4f}')
axes[0][0].annotate(f'Test: {fcn_acc*100:.2f}%',
                    xy=(10, fcn_acc), xytext=(8, fcn_acc-0.005),
                    fontsize=10, color='gray', fontweight='bold')

# Loss FCN
axes[0][1].plot(epochs, fcn_history.history['loss'],
                color='#E24B4A', lw=2.5, marker='o', ms=6, label='Train')
axes[0][1].plot(epochs, fcn_history.history['val_loss'],
                color='#E24B4A', lw=2.5, marker='s', ms=6,
                linestyle='--', alpha=0.7, label='Validation')
axes[0][1].set_title('FCN — Loss', fontweight='bold', fontsize=12)
axes[0][1].set_xlabel('Epoch'); axes[0][1].set_ylabel('Loss')
axes[0][1].legend()

# --- CNN ---
# Accuracy CNN
axes[1][0].plot(epochs, cnn_history.history['accuracy'],
                color='#1D9E75', lw=2.5, marker='o', ms=6, label='Train')
axes[1][0].plot(epochs, cnn_history.history['val_accuracy'],
                color='#1D9E75', lw=2.5, marker='s', ms=6,
                linestyle='--', alpha=0.7, label='Validation')
axes[1][0].set_title('CNN — Accuracy', fontweight='bold', fontsize=12)
axes[1][0].set_xlabel('Epoch'); axes[1][0].set_ylabel('Accuracy')
axes[1][0].legend()
axes[1][0].set_ylim(0.9, 1.01)
axes[1][0].annotate(f'Test: {cnn_acc*100:.2f}%',
                    xy=(10, cnn_acc), xytext=(8, cnn_acc-0.005),
                    fontsize=10, color='gray', fontweight='bold')

# Loss CNN
axes[1][1].plot(epochs, cnn_history.history['loss'],
                color='#BA7517', lw=2.5, marker='o', ms=6, label='Train')
axes[1][1].plot(epochs, cnn_history.history['val_loss'],
                color='#BA7517', lw=2.5, marker='s', ms=6,
                linestyle='--', alpha=0.7, label='Validation')
axes[1][1].set_title('CNN — Loss', fontweight='bold', fontsize=12)
axes[1][1].set_xlabel('Epoch'); axes[1][1].set_ylabel('Loss')
axes[1][1].legend()

# Fond coloré pour différencier
for ax in axes[0]: ax.set_facecolor('#F0F6FF')
for ax in axes[1]: ax.set_facecolor('#F0FFF8')

plt.suptitle('Courbes d\'apprentissage — FCN (bleu/rouge) vs CNN (vert/orange)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === VISUALISATION COMPARATIVE FINALE ===
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Bar chart accuracy
models_names = ['FCN', 'CNN']
accs = [fcn_acc*100, cnn_acc*100]
colors_m = ['#185FA5', '#1D9E75']
bars = axes[0].bar(models_names, accs, color=colors_m, edgecolor='white',
                   width=0.4)
axes[0].set_ylim(96, 100)
axes[0].set_title('Test Accuracy', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Accuracy (%)')
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()-0.2,
                 f'{val:.2f}%', ha='center', color='white',
                 fontweight='bold', fontsize=13)

# 2. Erreurs
errors = [int((1-a/100)*10000) for a in accs]
bars2 = axes[1].bar(models_names, errors, color=colors_m, edgecolor='white', width=0.4)
axes[1].set_title('Erreurs sur 10 000 images', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Nombre d\'erreurs')
for bar, val in zip(bars2, errors):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()/2,
                 str(val), ha='center', color='white', fontweight='bold', fontsize=13)

# 3. Val accuracy au fil des epochs (superposé)
axes[2].plot(epochs, fcn_history.history['val_accuracy'],
             color='#185FA5', lw=2.5, marker='o', ms=6, label=f'FCN  (test={fcn_acc:.4f})')
axes[2].plot(epochs, cnn_history.history['val_accuracy'],
             color='#1D9E75', lw=2.5, marker='s', ms=6, label=f'CNN  (test={cnn_acc:.4f})')
axes[2].fill_between(epochs,
                      fcn_history.history['val_accuracy'],
                      cnn_history.history['val_accuracy'],
                      alpha=0.2, color='#534AB7',
                      label='Écart CNN-FCN')
axes[2].set_title('Val Accuracy — FCN vs CNN par epoch', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Val Accuracy')
axes[2].legend(); axes[2].set_ylim(0.96, 1.01)

plt.suptitle('⚖️  Comparaison Finale — FCN vs CNN sur MNIST',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === MATRICES DE CONFUSION FCN vs CNN ===
y_pred_fcn = np.argmax(fcn_model.predict(X_test_flat, verbose=0), axis=1)
y_pred_cnn = np.argmax(cnn_model.predict(X_test_cnn,  verbose=0), axis=1)
y_true     = y_test_raw

cm_fcn = confusion_matrix(y_true, y_pred_fcn)
cm_cnn = confusion_matrix(y_true, y_pred_cnn)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, cm, title, acc in [
    (axes[0], cm_fcn, 'FCN', fcn_acc),
    (axes[1], cm_cnn, 'CNN', cnn_acc)
]:
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_pct, annot=True, fmt='.1%', cmap='Blues',
                vmin=0.85, vmax=1.0, ax=ax, linewidths=0.5,
                xticklabels=range(10), yticklabels=range(10))
    ax.set_title(f'{title} — Matrice de Confusion (% par classe)\nTest Accuracy = {acc*100:.2f}%',
                 fontweight='bold')
    ax.set_xlabel('Prédit'); ax.set_ylabel('Réel')

plt.suptitle('Matrices de Confusion Normalisées — FCN vs CNN',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Rapport complet
print('=== RAPPORT FCN ===')
print(classification_report(y_true, y_pred_fcn,
                             target_names=[str(i) for i in range(10)]))
print('=== RAPPORT CNN ===')
print(classification_report(y_true, y_pred_cnn,
                             target_names=[str(i) for i in range(10)]))

In [ ]:
# === ACCURACY PAR CHIFFRE : FCN vs CNN ===
per_class_fcn = np.diag(cm_fcn) / cm_fcn.sum(axis=1)
per_class_cnn = np.diag(cm_cnn) / cm_cnn.sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x = np.arange(10)
w = 0.35
axes[0].bar(x - w/2, per_class_fcn*100, w, color='#185FA5',
             edgecolor='white', label='FCN')
axes[0].bar(x + w/2, per_class_cnn*100, w, color='#1D9E75',
             edgecolor='white', label='CNN')
axes[0].set_ylim(94, 100.5)
axes[0].set_xticks(x); axes[0].set_xticklabels(range(10))
axes[0].set_title('Accuracy par chiffre — FCN vs CNN', fontweight='bold')
axes[0].set_xlabel('Chiffre'); axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
for i in range(10):
    axes[0].text(i-w/2, per_class_fcn[i]*100+0.1, f'{per_class_fcn[i]*100:.1f}',
                 ha='center', fontsize=7)
    axes[0].text(i+w/2, per_class_cnn[i]*100+0.1, f'{per_class_cnn[i]*100:.1f}',
                 ha='center', fontsize=7, color='#0F6E56')

# Amélioration CNN-FCN
diff = (per_class_cnn - per_class_fcn) * 100
colors_diff = ['#1D9E75' if d >= 0 else '#E24B4A' for d in diff]
axes[1].bar(range(10), diff, color=colors_diff, edgecolor='white')
axes[1].axhline(0, color='black', lw=1)
axes[1].set_xticks(range(10))
axes[1].set_title('Amélioration CNN vs FCN par chiffre (pp)', fontweight='bold')
axes[1].set_xlabel('Chiffre')
axes[1].set_ylabel('Différence (points de %)')
for i, d in enumerate(diff):
    axes[1].text(i, d + (0.05 if d >= 0 else -0.1),
                 f'{d:+.1f}', ha='center', fontsize=8, fontweight='bold')

plt.suptitle('Analyse par classe — FCN vs CNN', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 Accuracy par chiffre :')
print(f'  {"Chiffre":<10} {"FCN":>8} {"CNN":>8} {"Δ (CNN-FCN)":>12}')
print('-' * 45)
for d in range(10):
    delta = (per_class_cnn[d] - per_class_fcn[d]) * 100
    flag = '✅' if delta >= 0 else '🔻'
    print(f'  {flag} {d:<8} {per_class_fcn[d]*100:>7.2f}% {per_class_cnn[d]*100:>7.2f}%  {delta:>+8.2f}pp')

---
## 📋 Conclusion & Analyse comparative

### Résultats

| | FCN | CNN |
|---|---|---|
| Architecture | Input(784) → Dense(512) → Dense(256) → Output(10) | Conv2D(32) → Pool → Conv2D(64) → Pool → Flatten → Dense(128) → Output(10) |
| Paramètres | ~535 000 | ~421 000 |
| Test Accuracy | ~97-98% | ~99%+ |
| Epochs | 10 | 10 |

### Pourquoi le CNN fait-il mieux ?

1. **Localité** : les filtres Conv2D examinent des patches locaux (3×3) et détectent des features visuelles (bords, courbes, coins) indépendamment de leur position.
2. **Partage de poids** : un même filtre est appliqué partout sur l'image → moins de paramètres, moins d'overfitting.
3. **Hiérarchie** : Conv1 détecte les traits simples (bords), Conv2 combine ces traits en formes plus complexes (courbes de chiffres).
4. **MaxPooling** : réduit la résolution spatiale → robustesse aux petites translations et déformations.

### Le FCN, ses limites
Le FCN aplatit l'image dès le début et **perd toute information spatiale** : il ne sait plus que le pixel (3,3) est adjacent au pixel (3,4). Il doit apprendre cette structure depuis zéro avec beaucoup plus de paramètres.

### Quand utiliser l'un ou l'autre ?
| Cas | Modèle recommandé |
|---|---|
| Données tabulaires | FCN |
| Images, vidéos | CNN |
| Données séquentielles | RNN / LSTM |
| Texte, langage | Transformer |

---
*Notebook réalisé dans le cadre du DI Bootcamp — MNIST FCN vs CNN*